# Lab 11 — Deploy as a Web App (Gradio)
### *Turn your AI pipeline into a usable web app with guardrails and basic monitoring.*

<a href="https://colab.research.google.com/github/tulane-intro-ai-engineering/main/blob/main/labs/deployment_lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

---

## Overview
You will build a small deployed experience:
- a Python handler function that calls an LLM helper
- a Gradio UI that calls the handler
- guardrails (validation + caching + friendly errors)
- lightweight monitoring (request logs + latency averages)

Then you’ll launch the app from Colab and share the link.

---

## Learning goals
- Build and run a Gradio app from a notebook.
- Add input validation, caching, and friendly error handling.
- Log requests and compute a basic latency metric.
- Practice the scientific loop: change a deployment choice → observe latency and error rate.


In [1]:
# @title 🔧 Setup (Run this first)
!git clone --depth 1 -q https://github.com/tulane-intro-ai-engineering/main.git
import sys, time, random
import numpy as np
import pandas as pd

sys.path.append("/content/main")
from course_utils import lab11_setup, lab11_generate_reply, lab11_build_demo

lab11_setup()
random.seed(11)
np.random.seed(11)
print("✅ Environment ready!")


installing mermaid-python
Enter your OpenAI API key. It will only live in this Colab runtime.
OpenAI API key: ··········
✅ API key set.
installing dspy
✅ Environment ready!


## Pre-Lab Questions
Answer in 1–2 sentences each. (Edit this cell.)

1. What is one difference between a notebook demo and a deployed app?
2. Name one reason an AI app might be slow.
3. What is one thing you should never print to the screen in a deployed app?

Your answers:
1)  
2)  
3)


## Scientific Question & Hypothesis

**Question:**  
If we change **X = caching + input limits**, what happens to **Y = latency and error rate** in our app?

**My hypothesis:**  
I expect caching will _________ latency on repeated questions, and input limits will _________ error rate, because _________.

Write your hypothesis here:


## Scientific process plan
- **Question:** Do caching and input validation measurably improve the deployed experience?
- **Hypothesis:** written above
- **Experiment:** run the same question multiple times with cache OFF then ON; try a too-long input
- **Measurement:** latency (seconds) and % errors in request logs
- **Conclusion:** decide which guardrails you'd keep in a “real” deployment


# Part 1 — Build a minimal app handler
Your handler should:
1) validate input
2) optionally serve from cache
3) call the model helper
4) log the request

We provide the structure; you fill in a few TODOs.


In [ ]:
# @title ✅ TODO: Implement handler helpers (small)
request_logs = []
_cache = {}

def validate_text(user_text: str, max_chars: int = 1200) -> str:
    '''
    Return "OK" if valid, else return an error message string that starts with "ERROR:".

    TODO:
    - reject empty input
    - reject inputs longer than max_chars
    '''
    raise NotImplementedError("Implement validate_text")

def log_request(user_text: str, latency_s: float, cache_hit: bool, status: str):
    '''
    Append a dict into request_logs with:
      - time (HH:MM:SS)
      - input length
      - latency_s (rounded to 3 decimals)
      - cache_hit
      - status ("ok" or "error")
    '''
    raise NotImplementedError("Implement log_request")

def handler(user_text: str, use_cache: bool = True):
    '''
    The function Gradio will call. Returns a string answer (or friendly error).
    '''
    verdict = validate_text(user_text)
    if verdict != "OK":
        log_request(user_text, latency_s=0.0, cache_hit=False, status="error")
        return verdict

    if use_cache and user_text in _cache:
        ans = _cache[user_text]
        log_request(user_text, latency_s=0.0, cache_hit=True, status="ok")
        return ans

    t0 = time.perf_counter()
    ans = lab11_generate_reply(user_prompt=user_text)  # helper hides API details
    dt = time.perf_counter() - t0

    if use_cache:
        _cache[user_text] = ans

    log_request(user_text, latency_s=dt, cache_hit=False, status="ok")
    return ans


# Part 2 — Mini experiment: cache OFF vs ON
Run the same question multiple times and compare latency.


In [ ]:
# @title Experiment scaffold: cache OFF vs ON
question = "Explain RAG in 3 bullet points."

# cache OFF
request_logs.clear(); _cache.clear()
for _ in range(3):
    _ = handler(question, use_cache=False)
df_off = pd.DataFrame(request_logs)

# cache ON
request_logs.clear(); _cache.clear()
for _ in range(3):
    _ = handler(question, use_cache=True)
df_on = pd.DataFrame(request_logs)

print("Cache OFF:")
display(df_off)
print("Cache ON:")
display(df_on)

print("Average latency OFF:", round(df_off["latency_s"].mean(), 3))
print("Average latency ON :", round(df_on["latency_s"].mean(), 3))
print("Error rate ON:", round((df_on["status"]=='error').mean(), 3))


### Reflection
- Did caching help? By how much?
- What is one risk of caching in an AI app?

Write here:


# Part 3 — Launch the Gradio app
We will build the UI using a `course_utils` helper to keep code minimal.

Try:
- a normal question
- an empty input
- a huge input (copy/paste many characters)

Then look at the logs.


In [ ]:
# @title 🚀 Launch Gradio app
import gradio as gr

demo = lab11_build_demo(handler_fn=handler)
demo.launch(share=True)


# Part 4 — Deployment checklist (student writing)
Imagine you are shipping this app for real.

Write a checklist with 6–10 items:
- reliability (timeouts, retries, fallbacks)
- safety (refusal policy, injection defense)
- security (secrets, logging redaction)
- monitoring (what metrics alert you)

Write your checklist here:


# Optional: HF Spaces export
You can deploy the same Gradio app to Hugging Face Spaces.

**High-level steps:**
1. Create a new Space (Gradio).
2. Add an `app.py` that builds the UI.
3. Add `requirements.txt`.
4. Set secrets in Space settings (do not commit keys).




## Results
Summarize:
- cache OFF vs ON latency
- error behavior for invalid inputs
- what you logged and why

Write here:


## Conclusion
- Was your hypothesis supported?
- Which guardrails felt most important?
- What would you add next if you had another week?

Write here:


## Post-Lab Reflection
Answer briefly (2–4 sentences each). (Edit this cell.)

1. What surprised you about deploying an AI system as a web app?
2. What is one deployment risk you didn’t think about before this lab?
3. What is one monitoring metric you would alert on for your app?

Your answers:
1)  
2)  
3)


In [ ]:
# @title ✅ Checks for Lab 11
print("Running checks...")

try:
    v1 = validate_text("")
    assert isinstance(v1, str) and v1.startswith("ERROR")
    v2 = validate_text("hello")
    assert v2 == "OK"
    print("✅ validate_text ok")
except Exception as e:
    print("❌ validate_text failed:", e)

try:
    request_logs.clear()
    log_request("hi", 0.1234, False, "ok")
    assert len(request_logs) == 1
    assert "latency_s" in request_logs[0]
    print("✅ log_request ok")
except Exception as e:
    print("❌ log_request failed:", e)

try:
    out = handler("hello", use_cache=False)
    assert isinstance(out, str)
    print("✅ handler returns a string")
except Exception as e:
    print("❌ handler failed:", e)

print("Done.")
